# Section 1 - Environment set up

In [1]:
import pandas as pd
import numpy as np
import os

# Relative path for datasets
DATA_RAW_DIR = os.path.join("..", "data", "raw")
DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
salminen_path = os.path.join(DATA_RAW_DIR, "salminen_2022b.csv")
hollenbeck_path = os.path.join(DATA_RAW_DIR, "hollenbeck_2022.csv")

# Section 2 - Dataset A: Salminen et al. (2022b)

In [2]:
# Load and initial inspection
df_salminen = pd.read_csv(salminen_path)
print("Shape:", df_salminen.shape)
print("Columns:", df_salminen.columns.tolist())
print(df_salminen['label'].value_counts())
print(df_salminen.isnull().sum())

# Label encoding: standardise label: CG = Computer Generated (fake) -> 1, OR = Original (genuine) -> 0
df_salminen['is_fake'] = (df_salminen['label'] == 'CG').astype(int)
df_salminen['source'] = 'salminen'

Shape: (40432, 4)
Columns: ['category', 'rating', 'label', 'text_']
label
CG    20216
OR    20216
Name: count, dtype: int64
category    0
rating      0
label       0
text_       0
dtype: int64


# Section 3 - Dataset B: Hollenbeck et al. (2022)

In [3]:
# Load and initial inspection
df_hollenbeck_with_urls = pd.read_csv(hollenbeck_path)
print("Shape (raw):", df_hollenbeck_with_urls.shape)
print("Columns (raw):", df_hollenbeck_with_urls.columns.tolist())

C:\Users\admin\AppData\Local\Temp\ipykernel_21360\1454955210.py:2: DtypeWarning: Columns (0: fake_review_campaign_start_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df_hollenbeck_with_urls = pd.read_csv(hollenbeck_path)


Shape (raw): (381734, 23)
Columns (raw): ['asin', 'review_id', 'reviewer_id', 'review_title', 'review_text', 'review_rating', 'review_date', 'product_title', 'product_url', 'number_of_helpful', 'number_of_photos', 'photo_thumbnail_urls', 'photo_fullsize_urls', 'asin_url', 'review_url', 'reviewer_url', 'fake_review_campaign_start_date', 'fake_review_product', 'reviewer_classified_fake', 'reviewer_classified_honest', 'reviewer_labeled_fake', 'reviewer_labeled_honest', 'review_is_removed_by_amazon']


In [4]:
# Remove identifiable URL columns and unused columns
identifiable_columns = ['product_url', 'asin_url', 'review_url', 'reviewer_url']
non_essential_columns =  ['photo_thumbnail_urls', 'photo_fullsize_urls',
    'review_title', 'product_title', 'number_of_helpful', 'number_of_photos',
    'review_date', 'fake_review_campaign_start_date',
    'reviewer_classified_fake', 'reviewer_classified_honest',
    'review_is_removed_by_amazon']

# Drop the identifiable and non-essential columns from the Hollenbeck dataset
cols_to_drop = identifiable_columns + non_essential_columns
df_hollenbeck_clean = df_hollenbeck_with_urls.drop(columns=cols_to_drop)
print("Dropped:", cols_to_drop)
print("Shape (clean):", df_hollenbeck_clean.shape)


Dropped: ['product_url', 'asin_url', 'review_url', 'reviewer_url', 'photo_thumbnail_urls', 'photo_fullsize_urls', 'review_title', 'product_title', 'number_of_helpful', 'number_of_photos', 'review_date', 'fake_review_campaign_start_date', 'reviewer_classified_fake', 'reviewer_classified_honest', 'review_is_removed_by_amazon']
Shape (clean): (381734, 8)


In [6]:
# Build ground truth is_fake label
is_five_star = df_hollenbeck_clean['review_rating'] == 5
is_fake_product = df_hollenbeck_clean['fake_review_product'] == True
is_fake_reviewer = df_hollenbeck_clean['reviewer_labeled_fake'] == True

# Creating fake/real masks from the conditions above
# Fake: all three signals must align
fake_mask = is_five_star & is_fake_product & is_fake_reviewer
# Real: complement of fake_mask 
genuine_mask = ~fake_mask

# Percentage of fake vs genuine reviews
total = len(df_hollenbeck_clean)
print(f"Fake reviews: {fake_mask.sum()} ({fake_mask.sum()/total*100:.2f}%)")
print(f"Genuine reviews: {genuine_mask.sum()} ({genuine_mask.sum()/total*100:.2f}%)")

# Encode label and tag dataset source 
df_hollenbeck_labelled = df_hollenbeck_clean[fake_mask | genuine_mask].copy()
df_hollenbeck_labelled['is_fake'] = fake_mask[fake_mask | genuine_mask].astype(int)
df_hollenbeck_labelled['source'] = 'hollenbeck'

print("Shape (labelled):", df_hollenbeck_labelled.shape)
print(df_hollenbeck_labelled['is_fake'].value_counts())
print("Class ratio (genuine:fake):",
      f"{df_hollenbeck_labelled['is_fake'].value_counts()[0]
         / df_hollenbeck_labelled['is_fake'].value_counts()[1]:.1f} : 1")


Fake reviews: 20413 (5.35%)
Genuine reviews: 361321 (94.65%)
Shape (labelled): (381734, 10)
is_fake
0    361321
1     20413
Name: count, dtype: int64
Class ratio (genuine:fake): 17.7 : 1


# Section 4 - Save prepared datasets in CSV format

In [ ]:
# Save the cleaned datasets to the processed data directory
df_salminen.to_csv(os.path.join(DATA_PROCESSED_DIR, "salminen_clean.csv"), index=False)
df_hollenbeck_labelled.to_csv(os.path.join(DATA_PROCESSED_DIR, "hollenbeck_clean.csv"), index=False)

print("Saved to data/processed/:")
print(" - salminen_clean.csv   ", df_salminen.shape)
print(" - hollenbeck_clean.csv ", df_hollenbeck_labelled.shape)

Saved to data/processed/:
 - salminen_clean.csv    (40432, 6)
 - hollenbeck_clean.csv  (381734, 10)
